# Poketeer Card Embedder Training

Trains a ResNet50-based card embedder with **in-batch hard negative mining** for accurate Pokemon card identification.

**Runtime → Change runtime type → GPU (T4)**

In [ ]:
# 1. Check GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > GPU')

In [ ]:
# 2. Download card catalog from Supabase
!pip install -q supabase

SUPABASE_URL = "https://bohtisqptjwbmlfcnliz.supabase.co"
# Anon key is fine here — cards table has public read via RLS
SUPABASE_ANON_KEY = "sb_publishable_ge1aLD1g3od3jjzBe8vTpQ_qvpB3Gfw"

from supabase import create_client
sb = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)

# Fetch all cards (paginated — Supabase limits to 1000 per request)
all_cards = []
page_size = 1000
offset = 0
while True:
    res = sb.table('cards').select('id, name, number, set_id, rarity, image_small, image_large, supertype, subtypes, hp, artist, types').range(offset, offset + page_size - 1).execute()
    batch = res.data or []
    all_cards.extend(batch)
    if len(batch) < page_size:
        break
    offset += page_size

print(f'Loaded {len(all_cards)} cards from Supabase')

In [ ]:
# 3. Download all card images (cached to disk)
import os
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

IMG_DIR = Path('card_images')
IMG_DIR.mkdir(exist_ok=True)

def download_image(card):
    url = card.get('image_small', '')
    if not url:
        return None
    path = IMG_DIR / f"{card['id']}.png"
    if path.exists():
        return str(path)
    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        path.write_bytes(r.content)
        return str(path)
    except:
        return None

# Download with 16 threads
card_paths = {}
with ThreadPoolExecutor(max_workers=16) as pool:
    futures = {pool.submit(download_image, c): c['id'] for c in all_cards}
    for f in tqdm(as_completed(futures), total=len(futures), desc='Downloading'):
        card_id = futures[f]
        path = f.result()
        if path:
            card_paths[card_id] = path

print(f'Downloaded {len(card_paths)} / {len(all_cards)} card images')

In [ ]:
# 4. Model definition
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

class CardEmbedder(nn.Module):
    """ResNet50 backbone + projection head for metric learning."""
    def __init__(self, embed_dim=512, freeze_backbone_layers=6):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT
        backbone = models.resnet50(weights=weights)
        children = list(backbone.children())
        for i, child in enumerate(children[:freeze_backbone_layers]):
            for param in child.parameters():
                param.requires_grad = False
        self.backbone = nn.Sequential(*children[:-1])
        self.flatten = nn.Flatten()
        self.projector = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(1024, embed_dim),
        )

    def forward(self, x):
        features = self.backbone(x)
        features = self.flatten(features)
        embeddings = self.projector(features)
        return F.normalize(embeddings, p=2, dim=1)

print('Model defined')

In [ ]:
# 5. Dataset with strong augmentations
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import random

# Strong augmentation to simulate real phone photos
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomPerspective(distortion_scale=0.25, p=0.7),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
    transforms.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

class CardDataset(Dataset):
    """Returns 2 augmented views of each card for in-batch contrastive learning."""
    def __init__(self, cards, card_paths, transform):
        self.cards = [c for c in cards if c['id'] in card_paths]
        self.card_paths = card_paths
        self.transform = transform

    def __len__(self):
        return len(self.cards)

    def __getitem__(self, idx):
        card = self.cards[idx]
        path = self.card_paths[card['id']]
        img = Image.open(path).convert('RGB')
        # Return 2 different augmented views of the same card
        view1 = self.transform(img)
        view2 = self.transform(img)
        return view1, view2, idx

dataset = CardDataset(all_cards, card_paths, train_transform)
print(f'Dataset: {len(dataset)} cards with images')

In [ ]:
# 6. Training with in-batch hard negative mining
import numpy as np
from tqdm.notebook import tqdm

def batch_hard_triplet_loss(embeddings, labels, margin=0.5):
    """
    For each anchor, find:
    - hardest positive (same label, farthest embedding)
    - hardest negative (different label, closest embedding)
    This forces the model to learn fine-grained distinctions.
    """
    # Pairwise distance matrix
    dist = torch.cdist(embeddings, embeddings, p=2)  # (B, B)
    B = embeddings.size(0)

    # Labels match matrix
    label_eq = labels.unsqueeze(0) == labels.unsqueeze(1)  # (B, B)

    # Hardest positive: max distance among same-label pairs
    pos_dist = dist.clone()
    pos_dist[~label_eq] = 0  # zero out different labels
    hardest_pos, _ = pos_dist.max(dim=1)  # (B,)

    # Hardest negative: min distance among different-label pairs
    neg_dist = dist.clone()
    neg_dist[label_eq] = float('inf')  # ignore same labels
    hardest_neg, _ = neg_dist.min(dim=1)  # (B,)

    # Triplet loss
    losses = F.relu(hardest_pos - hardest_neg + margin)
    return losses.mean()

# Training config
EPOCHS = 30
BATCH_SIZE = 64  # Larger batch = more negatives to mine from
LR = 1e-4
MARGIN = 0.5
EMBED_DIM = 512

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = CardEmbedder(embed_dim=EMBED_DIM, freeze_backbone_layers=4).to(device)  # Unfreeze more layers
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

print(f'Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LR}, margin={MARGIN}')
print(f'Device: {device}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print()

best_loss = float('inf')
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    n_batches = 0

    pbar = tqdm(loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for view1, view2, indices in pbar:
        # Stack both views: [v1_0, v1_1, ..., v2_0, v2_1, ...]
        images = torch.cat([view1, view2], dim=0).to(device)
        # Labels: each card appears twice (same index = same card)
        labels = torch.cat([indices, indices], dim=0).to(device)

        embeddings = model(images)
        loss = batch_hard_triplet_loss(embeddings, labels, margin=MARGIN)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    scheduler.step()
    avg_loss = total_loss / max(n_batches, 1)
    print(f'  Epoch {epoch+1}: avg_loss={avg_loss:.4f}, lr={scheduler.get_last_lr()[0]:.6f}')

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), 'card_classifier.pt')
        print(f'  -> Saved best model (loss={best_loss:.4f})')

print(f'\nTraining complete! Best loss: {best_loss:.4f}')

In [ ]:
# 7. Rebuild embeddings index with the new model
print('Rebuilding embeddings with trained model...')

model.load_state_dict(torch.load('card_classifier.pt', map_location=device))
model.eval()

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Process in batches for speed
EVAL_BATCH = 64
all_embeddings = []
all_ids = []
valid_cards = [c for c in all_cards if c['id'] in card_paths]

for i in tqdm(range(0, len(valid_cards), EVAL_BATCH), desc='Embedding'):
    batch_cards = valid_cards[i:i+EVAL_BATCH]
    tensors = []
    batch_ids = []
    for card in batch_cards:
        try:
            img = Image.open(card_paths[card['id']]).convert('RGB')
            tensors.append(eval_transform(img))
            batch_ids.append(card['id'])
        except:
            continue

    if not tensors:
        continue

    batch_tensor = torch.stack(tensors).to(device)
    with torch.inference_mode():
        embs = model(batch_tensor).cpu().numpy()
    all_embeddings.append(embs)
    all_ids.extend(batch_ids)

embeddings_matrix = np.concatenate(all_embeddings, axis=0)
print(f'Generated {len(all_ids)} embeddings, shape: {embeddings_matrix.shape}')

In [ ]:
# 8. Quick validation — test with a known card
from PIL import Image as PILImage

# Pick a random card and check its top matches
test_idx = random.randint(0, len(all_ids) - 1)
test_id = all_ids[test_idx]
test_emb = embeddings_matrix[test_idx]
test_card = next(c for c in all_cards if c['id'] == test_id)

# Cosine similarity against all
sims = embeddings_matrix @ test_emb  # already L2-normalized
top_indices = np.argsort(sims)[::-1][:10]

print(f'Test card: {test_card["name"]} ({test_id})')
print(f'\nTop 10 matches:')
for rank, idx in enumerate(top_indices):
    match_id = all_ids[idx]
    match_card = next(c for c in all_cards if c['id'] == match_id)
    print(f'  {rank+1}. {sims[idx]:.4f}  {match_id:20s}  {match_card["name"]}')

# Check score spread
print(f'\nScore spread: top1={sims[top_indices[0]]:.4f}, top5={sims[top_indices[4]]:.4f}, top10={sims[top_indices[9]]:.4f}')
print(f'Mean similarity: {sims.mean():.4f}, Std: {sims.std():.4f}')

In [ ]:
# 9. Upload new model and embeddings to Supabase
# Uses service key — paste yours here
SUPABASE_SERVICE_KEY = input('Paste your Supabase SERVICE KEY: ')

sb_admin = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

BATCH = 100
for i in tqdm(range(0, len(all_ids), BATCH), desc='Uploading embeddings'):
    rows = []
    for j in range(i, min(i + BATCH, len(all_ids))):
        vec = embeddings_matrix[j].tolist()
        vec_str = '[' + ','.join(f'{v:.6f}' for v in vec) + ']'
        rows.append({'card_id': all_ids[j], 'embedding': vec_str})
    sb_admin.table('card_embeddings').upsert(rows).execute()

print(f'Uploaded {len(all_ids)} embeddings to Supabase')

In [ ]:
# 10. Download the trained model
from google.colab import files
files.download('card_classifier.pt')
print('Download card_classifier.pt and upload it to your HF Space')